# hA2025 reweight: new production branch vs `src/pion_fsi_reweighting.py`

The new Run 4b BNB $\nu$ overlay checkout (`..._hist_reweight.root`) has a `hA2025reweight` branch in
`nuselection/NeutrinoSelectionFilter`. Here we check it against our pure-Python hA2018→hA2025 calculation.
For reference we also check the older `..._hist_FSIrwgt.root` file, made with the `rwgtNtupleFSI.C` macro.

Our calculator has two variants:
- **default** (`hN_charge_correction=True`): FracADep 2025/2018 table ratio × GENIE's official pi0-only
  "hN charge correction" (`HAIntranuke2025::HadronFateHA`). This is the weight in `hA2025_pion_fsi_rw_weight`.
- **macro** (`hN_charge_correction=False`): table ratio only. This is what `rwgtNtupleFSI.C` computes.

In [ ]:
import sys, numpy as np, uproot, matplotlib.pyplot as plt
sys.path.insert(0, "../src")
from pion_fsi_reweighting import compute_pion_fsi_weights_from_arrays

D = "/nevis/riverside/data/leehagaman/ngem/other_files/checkout_MCC9.10_Run4b_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist"
FILES = {"new (reweight)": D + "_reweight.root", "old (FSIrwgt macro)": D + "_FSIrwgt.root"}
BR = ["mc_generator_pdg", "mc_generator_mother", "mc_generator_rescatter", "mc_generator_statuscode",
      "mc_generator_E", "mc_generator_px", "mc_generator_py", "mc_generator_pz"]

res = {}
for name, path in FILES.items():
    with uproot.open(path) as f:
        a = f["nuselection/NeutrinoSelectionFilter"].arrays(BR + ["hA2025reweight", "run", "sub", "evt"], library="np")
        pot = f["nuselection/SubRun"]["pot"].array(library="np").sum()
    args = [a[b] for b in BR]
    w_on, _, q = compute_pion_fsi_weights_from_arrays(*args, hN_charge_correction=True, return_queries=True)
    w_off, _ = compute_pion_fsi_weights_from_arrays(*args, hN_charge_correction=False)
    res[name] = dict(branch=a["hA2025reweight"].astype(float), w_on=w_on, w_off=w_off, q=q, pot=pot,
                     key=a["run"].astype(np.int64) * 10**10 + a["sub"].astype(np.int64) * 10**5 + a["evt"])
    del a, args

## 1. Which variant does each branch match?

In [ ]:
print(f"{'file':22s} {'N':>8s} {'POT':>10s} | {'variant':8s} {'|d|<1e-6':>10s} {'max|d|':>9s} {'sum(ours)/sum(branch)':>22s}")
for name, r in res.items():
    for v in ["w_on", "w_off"]:
        d = r[v] - r["branch"]
        print(f"{name:22s} {len(d):8d} {r['pot']:10.4g} | {'default' if v=='w_on' else 'macro':8s} "
              f"{np.mean(np.abs(d)<1e-6)*100:9.4f}% {np.abs(d).max():9.3g} {r[v].sum()/r['branch'].sum():22.7f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (name, r) in zip(axs, res.items()):
    for v, lab in [("w_on", "default (hN corr.)"), ("w_off", "macro (no hN corr.)")]:
        d = r[v] - r["branch"]
        ax.hist(np.clip(d, -1, 1), bins=np.linspace(-1, 1, 201), histtype="step", lw=1.5, label=lab)
    ax.set_yscale("log"); ax.set_title(f"branch in {name}"); ax.set_xlabel("ours − hA2025reweight (clipped ±1)")
    ax.legend()
plt.tight_layout()

## 2. Where the new branch differs from the macro
If the new production just added GENIE's hN charge correction, then new-branch ≠ macro should happen
**exactly** on the events with an FSI'd pi0 (charged pions are untouched by that correction).

In [ ]:
r = res["new (reweight)"]; q = r["q"]
has_pi0 = np.zeros(len(r["branch"]), bool); has_pi0[q["evt"][q["pdg"] == 111]] = True
differs = np.abs(r["branch"] - r["w_off"]) > 1e-6
print("events where new branch != macro value :", differs.sum())
print("  ... of which have an FSI'd pi0        :", (differs & has_pi0).sum())
print("FSI'd-pi0 events where it does NOT differ:", (has_pi0 & ~differs).sum())
print(f"total-weight effect of the hN correction: sum(new branch)/sum(macro) = {r['branch'].sum()/r['w_off'].sum():.5f}")

## 3. Same events in both files
The old macro file matched our macro variant on all but ~54 events. We put that down to a Triangle
tie-break on the degenerate A grid. Here we check those events in the new file, and confirm the inputs are identical.

In [ ]:
o, n = res["old (FSIrwgt macro)"], res["new (reweight)"]
pos = {k: i for i, k in enumerate(o["key"])}
in_old = np.array([k in pos for k in n["key"]]); io = np.array([pos[k] for k in n["key"][in_old]])
print("new events also in old file:", in_old.sum(), "/", len(n["key"]), "; old-only events:", len(o["key"]) - in_old.sum())
print("our macro weight identical across files (same mc_generator inputs):", np.all(o["w_off"][io] == n["w_off"][in_old]))
bad_old = np.abs(o["w_off"] - o["branch"]) > 1e-6
bad_keys = o["key"][bad_old]; m = np.isin(n["key"], bad_keys)
print(f"old-file residual events: {bad_old.sum()}, present in new file: {m.sum()}, "
      f"max |default - new branch| on them: {np.abs(n['w_on'][m]-n['branch'][m]).max():.2g}")

In [ ]:
# The old-only events: are they whole subruns (with their POT) or scattered events?
missing = set((o["key"] // 10**5).tolist()) - rn
n_ev_missing = np.isin(o["key"] // 10**5, list(missing)).sum()
print(f"subruns missing from new file: {len(missing)}, holding {n_ev_missing} events (= all old-only events: {n_ev_missing == len(o['key']) - in_old.sum()})")
print(f"POT old {o['pot']:.5g}  new {n['pot']:.5g}  ratio {n['pot']/o['pot']:.4f};  events ratio {len(n['key'])/len(o['key']):.4f}")

## Summary
See the printed numbers above. The conclusions (also in the final cell) are:
- The new `hA2025reweight` branch equals our **default** weight (`hA2025_pion_fsi_rw_weight`, including the hN
  charge correction) to float32 precision on every event, including the 54 events that were Triangle tie-break
  residuals against the old macro.
- The old/new differences come entirely from the hN charge correction. They happen exactly on the events with an
  FSI'd pi0, and lower the total weight by about 0.19%.
- The new file is a subset of the old one, missing whole subruns together with their POT, so normalization stays
  consistent.